# MotoGP Grand Prix Events Held - Dataset Exploration

This notebook explores the Grand Prix events dataset showing how many times each track has hosted MotoGP races across different countries.

## 0. Notebook Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading

Let's load the grand_prix_events_held.csv file containing track hosting statistics.

In [ ]:
# Load data from CSV
data_path = Path("../data/raw/grand_prix_events_held.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 10 rows:")
df.head(10)

## 2. Data Structure Exploration

Let's examine the data structure and understand the hosting patterns.

In [ ]:
print(f"Dimensions: {df.shape}")
print("-" * 50)
print(f"Columns: {list(df.columns)}")
print("-" * 50)
print(f"\nData types:")
print(df.dtypes)
print("-" * 50)
print(f"\nNull values:")
print(df.isnull().sum())

In [ ]:
# Basic statistics
print("=== BASIC STATISTICS ===")
print(f"Total unique tracks: {df['Track'].nunique()}")
print(f"Total unique countries: {df['Country'].nunique()}")
print(f"Total Grand Prix events: {df['Times'].sum()}")
print(f"\nEvents per track statistics:")
print(df['Times'].describe())

## 3. Track Analysis

Let's analyze which tracks have hosted the most Grand Prix events.

In [ ]:
# Top tracks by number of events hosted
print("=== TOP TRACKS BY EVENTS HOSTED ===")
top_tracks = df.nlargest(15, 'Times')
print(top_tracks[['Track', 'Country', 'Times']])

# Bar chart of top 15 tracks
plt.figure(figsize=(14, 8))
plt.bar(range(len(top_tracks)), top_tracks['Times'], color='skyblue')
plt.xticks(range(len(top_tracks)), [f"{track[:20]}..." if len(track) > 20 else track 
                                   for track in top_tracks['Track']], rotation=45, ha='right')
plt.xlabel('Track')
plt.ylabel('Number of Events Hosted')
plt.title('Top 15 Tracks by Number of Grand Prix Events Hosted')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of events hosted
plt.figure(figsize=(12, 6))
plt.hist(df['Times'], bins=20, alpha=0.7, color='lightcoral', edgecolor='black')
plt.xlabel('Number of Events Hosted')
plt.ylabel('Number of Tracks')
plt.title('Distribution of Events Hosted Across All Tracks')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Categories of tracks
print("\n=== TRACK CATEGORIES BY FREQUENCY ===")
def categorize_track_frequency(times):
    if times >= 100:
        return 'Historic Venue (100+ events)'
    elif times >= 50:
        return 'Regular Venue (50-99 events)'
    elif times >= 20:
        return 'Frequent Venue (20-49 events)'
    elif times >= 10:
        return 'Occasional Venue (10-19 events)'
    else:
        return 'Rare Venue (1-9 events)'

df['Track Category'] = df['Times'].apply(categorize_track_frequency)
category_counts = df['Track Category'].value_counts()
print(category_counts)

## 4. Country Analysis

Let's analyze which countries have hosted the most Grand Prix events.

In [ ]:
# Countries by total events hosted
print("=== COUNTRIES BY TOTAL EVENTS HOSTED ===")
country_totals = df.groupby('Country').agg({
    'Times': 'sum',
    'Track': 'count'
}).rename(columns={'Track': 'Number_of_Tracks'})
country_totals = country_totals.sort_values('Times', ascending=False)

print("Top 15 countries:")
print(country_totals.head(15))

# Bar chart of top countries
plt.figure(figsize=(14, 8))
top_15_countries = country_totals.head(15)
plt.bar(range(len(top_15_countries)), top_15_countries['Times'], color='lightgreen')
plt.xticks(range(len(top_15_countries)), top_15_countries.index, rotation=45)
plt.xlabel('Country')
plt.ylabel('Total Events Hosted')
plt.title('Top 15 Countries by Total Grand Prix Events Hosted')
plt.tight_layout()
plt.show()

In [ ]:
# Countries by number of tracks
print("=== COUNTRIES BY NUMBER OF TRACKS ===")
country_tracks = country_totals.sort_values('Number_of_Tracks', ascending=False)
print("Top 15 countries by number of different tracks:")
print(country_tracks.head(15))

# Scatter plot: Number of tracks vs Total events
plt.figure(figsize=(12, 8))
plt.scatter(country_totals['Number_of_Tracks'], country_totals['Times'], 
           alpha=0.7, s=80, color='purple')

# Add labels for notable countries
for country in country_totals.head(10).index:
    x = country_totals.loc[country, 'Number_of_Tracks']
    y = country_totals.loc[country, 'Times']
    plt.annotate(country, (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('Number of Different Tracks')
plt.ylabel('Total Events Hosted')
plt.title('Country Analysis: Number of Tracks vs Total Events Hosted')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Continental Analysis

Let's analyze the geographical distribution by continents.

In [ ]:
# Create continent mapping
continent_mapping = {
    'NL': 'Europe', 'CZ': 'Europe', 'BE': 'Europe', 'DE': 'Europe', 'ES': 'Europe',
    'IT': 'Europe', 'FR': 'Europe', 'GB': 'Europe', 'AT': 'Europe', 'FI': 'Europe',
    'SE': 'Europe', 'CH': 'Europe', 'PT': 'Europe', 'IE': 'Europe', 'NO': 'Europe',
    'DK': 'Europe', 'HU': 'Europe', 'SI': 'Europe', 'SK': 'Europe', 'HR': 'Europe',
    'JP': 'Asia', 'MY': 'Asia', 'TH': 'Asia', 'CN': 'Asia', 'IN': 'Asia', 'ID': 'Asia',
    'KR': 'Asia', 'TW': 'Asia', 'PH': 'Asia', 'SG': 'Asia', 'VN': 'Asia', 'QA': 'Asia',
    'AE': 'Asia', 'SA': 'Asia', 'IL': 'Asia', 'TR': 'Asia', 'LB': 'Asia',
    'AU': 'Oceania', 'NZ': 'Oceania',
    'US': 'North America', 'CA': 'North America', 'MX': 'North America',
    'BR': 'South America', 'AR': 'South America', 'VE': 'South America',
    'ZA': 'Africa', 'EG': 'Africa', 'MA': 'Africa', 'TN': 'Africa'
}

df['Continent'] = df['Country'].map(continent_mapping)
df['Continent'] = df['Continent'].fillna('Other')

print("=== CONTINENTAL ANALYSIS ===")
continent_stats = df.groupby('Continent').agg({
    'Times': 'sum',
    'Track': 'count',
    'Country': 'nunique'
}).rename(columns={'Track': 'Number_of_Tracks', 'Country': 'Number_of_Countries'})
continent_stats = continent_stats.sort_values('Times', ascending=False)

print("Events hosted by continent:")
print(continent_stats)

# Pie chart of continental distribution
plt.figure(figsize=(10, 8))
plt.pie(continent_stats['Times'], labels=continent_stats.index, 
        autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Grand Prix Events by Continent')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 6. Track Hosting Patterns

Let's analyze patterns in track hosting frequency.

In [ ]:
# Track frequency categories visualization
plt.figure(figsize=(10, 8))
category_counts = df['Track Category'].value_counts()
plt.pie(category_counts.values, labels=category_counts.index, 
        autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Tracks by Hosting Frequency Category')
plt.axis('equal')
plt.tight_layout()
plt.show()

print("\n=== TRACK FREQUENCY STATISTICS ===")
for category in category_counts.index:
    category_data = df[df['Track Category'] == category]
    total_events = category_data['Times'].sum()
    avg_events = category_data['Times'].mean()
    print(f"{category}: {len(category_data)} tracks, {total_events} total events, {avg_events:.1f} avg per track")

In [ ]:
# Historic venues analysis
print("=== HISTORIC VENUES ANALYSIS ===")
historic_venues = df[df['Track Category'] == 'Historic Venue (100+ events)']
if len(historic_venues) > 0:
    print("Historic venues (100+ events hosted):")
    print(historic_venues[['Track', 'Country', 'Times']].sort_values('Times', ascending=False))
else:
    print("No venues have hosted 100+ events.")

# Regular venues analysis  
print("\n=== REGULAR VENUES ANALYSIS ===")
regular_venues = df[df['Track Category'] == 'Regular Venue (50-99 events)']
if len(regular_venues) > 0:
    print("Regular venues (50-99 events hosted):")
    print(regular_venues[['Track', 'Country', 'Times']].sort_values('Times', ascending=False))
else:
    print("No venues have hosted 50-99 events.")

## 7. Average Events per Track by Country

Let's analyze the average hosting intensity by country.

In [ ]:
# Calculate average events per track by country
print("=== AVERAGE EVENTS PER TRACK BY COUNTRY ===")
country_averages = df.groupby('Country').agg({
    'Times': ['sum', 'mean', 'count']
})
country_averages.columns = ['Total_Events', 'Avg_Events_Per_Track', 'Number_of_Tracks']
country_averages = country_averages.sort_values('Avg_Events_Per_Track', ascending=False)

print("Top 15 countries by average events per track:")
print(country_averages.head(15))

# Bar chart of average events per track
plt.figure(figsize=(14, 8))
top_15_avg = country_averages.head(15)
plt.bar(range(len(top_15_avg)), top_15_avg['Avg_Events_Per_Track'], color='orange')
plt.xticks(range(len(top_15_avg)), top_15_avg.index, rotation=45)
plt.xlabel('Country')
plt.ylabel('Average Events Per Track')
plt.title('Top 15 Countries by Average Events Per Track')
plt.tight_layout()
plt.show()

## 8. Hosting Concentration Analysis

Let's analyze how concentrated track hosting is globally.

In [ ]:
# Calculate hosting concentration
print("=== HOSTING CONCENTRATION ANALYSIS ===")

# Top tracks concentration
total_events = df['Times'].sum()
top_5_events = df.nlargest(5, 'Times')['Times'].sum()
top_10_events = df.nlargest(10, 'Times')['Times'].sum()
top_20_events = df.nlargest(20, 'Times')['Times'].sum()

print(f"Total events across all tracks: {total_events}")
print(f"Top 5 tracks host: {top_5_events} events ({top_5_events/total_events*100:.1f}%)")
print(f"Top 10 tracks host: {top_10_events} events ({top_10_events/total_events*100:.1f}%)")
print(f"Top 20 tracks host: {top_20_events} events ({top_20_events/total_events*100:.1f}%)")

# Countries concentration
top_5_countries = country_totals.head(5)['Times'].sum()
top_10_countries = country_totals.head(10)['Times'].sum()

print(f"\nTop 5 countries host: {top_5_countries} events ({top_5_countries/total_events*100:.1f}%)")
print(f"Top 10 countries host: {top_10_countries} events ({top_10_countries/total_events*100:.1f}%)")

# Lorenz curve for track concentration
sorted_times = df.sort_values('Times')['Times'].values
cumsum_times = np.cumsum(sorted_times)
cumsum_normalized = cumsum_times / cumsum_times[-1]
tracks_normalized = np.arange(1, len(sorted_times) + 1) / len(sorted_times)

plt.figure(figsize=(10, 8))
plt.plot(tracks_normalized, cumsum_normalized, linewidth=2, label='Actual Distribution')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Equality')
plt.xlabel('Cumulative Proportion of Tracks')
plt.ylabel('Cumulative Proportion of Events')
plt.title('Lorenz Curve: Distribution of Events Across Tracks')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Key Insights Summary

Based on the exploration of the Grand Prix events hosted dataset:

### Dataset Characteristics
- Contains hosting statistics for racing venues worldwide
- Shows clear hierarchy of venue importance in MotoGP history
- Covers multiple continents with varying levels of participation

### Track Hosting Patterns
- **Historic Venues**: TT Circuit Assen leads with most events hosted
- **Regular Venues**: Several European circuits dominate hosting frequency
- **Occasional Venues**: Many tracks host only a limited number of events
- **Distribution**: Highly concentrated with few tracks hosting majority of events

### Geographical Patterns
- **European Dominance**: Europe hosts the majority of Grand Prix events
- **Multi-Track Countries**: Some countries have multiple hosting venues
- **Single-Venue Countries**: Many countries host through a single circuit
- **Continental Spread**: Representation across all continents but uneven distribution

### Hosting Concentration
- Top tracks host disproportionately large share of total events
- Geographic concentration in certain regions (particularly Europe)
- Clear distinction between historic venues and occasional hosts
- Lorenz curve shows significant inequality in hosting distribution

### Strategic Insights
- Some circuits are clearly cornerstone venues for the championship
- Geographic expansion has occurred but Europe remains the center
- Track hosting frequency correlates with historical importance
- Continental representation varies significantly

This exploration provides the foundation for answering specific business questions about circuit hosting patterns, continental representation, and geographic distribution of MotoGP events.